# 🐍 Day 28 · Python Basics — Client–Server, XML-RPC & base64 (Remote MCP with FreeCAD)

> **Goal of this notebook:** The FreeCAD project sends Python commands from a server to the FreeCAD app and gets screenshots back. Learn the client–server idea, run an RPC server yourself and send images as base64 text.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

📦 **First** do *Step 1 - Install today's packages* in [`00_START_HERE.md`](00_START_HERE.md).

✅ No API key needed for this notebook.

### 📋 What you will learn
1. What client and server mean
2. Running code in the background with a thread
3. An XML-RPC server and client
4. Bytes vs text
5. base64 — sending images as text

## 1. Client and server
- **Server** = a program that *waits* for requests and answers them (FreeCAD bridge).
- **Client** = a program that *sends* requests (the MCP server / Claude).
- **RPC** (Remote Procedure Call) = *call a function that lives in another program* as if it were local.

Read `xmlrpc_demo/rpc_vs_rest.md` in this folder for the full comparison.

## 2. Run the server in the background
A server waits forever, so we start it in a **thread** — a second worker running next to the notebook.

In [ ]:
from xmlrpc.server import SimpleXMLRPCServer
import threading

def make_box(length, width, height):
    # FreeCAD would create a real 3D box here
    return f"Created box {length}x{width}x{height} (volume {length * width * height})"

def factorial(n):
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result

server = SimpleXMLRPCServer(("127.0.0.1", 0), logRequests=False)   # port 0 = pick any free port
server.register_function(make_box)
server.register_function(factorial)
port = server.server_address[1]

thread = threading.Thread(target=server.serve_forever, daemon=True)   # daemon = stops with the notebook
thread.start()
print(f"🖥️ Server running on port {port}")

## 3. The client calls functions on the server

In [ ]:
from xmlrpc.client import ServerProxy

client = ServerProxy(f"http://127.0.0.1:{port}")

print(client.make_box(10, 20, 5))        # looks local, but runs inside the server!
print("10! =", client.factorial(10))

In [ ]:
server.shutdown()          # stop the server when we are done
print("Server stopped ✅")

## 4. Bytes vs text
Images, audio and files are **bytes** (raw numbers 0–255). Text is a **string**.
`.encode()` turns text into bytes, `.decode()` turns bytes back into text.

In [ ]:
text = "FreeCAD ✓"
data = text.encode("utf-8")
print(data, type(data), len(data), "bytes")
print(data.decode("utf-8"))

## 5. 🔗 base64 — images as text
JSON can only carry text, so screenshots are converted to **base64** text first, then decoded back into bytes on the other side.

In [ ]:
import base64

fake_png = bytes([137, 80, 78, 71, 13, 10, 26, 10]) + b"pretend image data"   # PNG files start with these 8 bytes

encoded = base64.b64encode(fake_png).decode("ascii")    # bytes -> base64 text
print("Sent as text:", encoded)

decoded = base64.b64decode(encoded)                     # text -> the original bytes
print("Same bytes back?", decoded == fake_png)

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Register a new function `add(a, b)` on a new server and call it from a client.
2. Encode the text `"hello"` to base64 and decode it back.
3. What is the length in bytes of `"नमस्ते"` when encoded as UTF-8?

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
srv = SimpleXMLRPCServer(("127.0.0.1", 0), logRequests=False)
srv.register_function(lambda a, b: a + b, "add")
threading.Thread(target=srv.serve_forever, daemon=True).start()
print(ServerProxy(f"http://127.0.0.1:{srv.server_address[1]}").add(2, 3))
srv.shutdown()

e = base64.b64encode(b"hello").decode()
print(e, base64.b64decode(e))
print(len("नमस्ते".encode("utf-8")))   # 18 bytes
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| Server | `SimpleXMLRPCServer(('127.0.0.1', 0))` |
| Expose a function | `server.register_function(make_box)` |
| Background thread | `threading.Thread(target=f, daemon=True).start()` |
| Client call | `ServerProxy(url).make_box(1, 2, 3)` |
| Text <-> bytes | `s.encode(), b.decode()` |
| base64 | `base64.b64encode(data)` |

## ➡️ What's next?
You now know the Python used in **Day_28_Remote_MCP_FreeCAD**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀